In [ ]:
import time
import tracemalloc
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import numpy as np
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['savefig.edgecolor'] = 'white'

plt.rcParams['figure.dpi'] = 500
plt.rcParams['savefig.dpi'] = 500
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

from pathlib import Path
import importlib.util
import sys
import tempfile
import urllib.request

_reference_util_path = next(
    (
        candidate
        for base in (Path.cwd(), *Path.cwd().parents)
        for candidate in (
            base / "capitulo4" / "referencias" / "util.py",
            base / "referencias" / "util.py",
        )
        if candidate.exists()
    ),
    None,
)
if _reference_util_path is None:
    _reference_util_path = Path(tempfile.gettempdir()) / "capitulo4_reference_util.py"
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Notas-a-Mano-serie-de-libros/"
        "3_notas-a-mano-sobre-analisis-de-complejidad-computacional/"
        "main/capitulo4/referencias/util.py",
        _reference_util_path,
    )
_reference_spec = importlib.util.spec_from_file_location(
    "capitulo4_reference_util", _reference_util_path
)
_reference_util = importlib.util.module_from_spec(_reference_spec)
sys.modules[_reference_spec.name] = _reference_util
_reference_spec.loader.exec_module(_reference_util)
graficar_complejidad = _reference_util.graficar_complejidad
modelo_constante = _reference_util.modelo_constante
modelo_lineal = _reference_util.modelo_lineal
modelo_cuadratico = _reference_util.modelo_cuadratico


In [ ]:
def medir_tiempo_iterar(func, n_iter):
    tiempos = np.zeros(n_iter)
    for i in range(n_iter):
        inicio = time.perf_counter()
        func()
        tiempos[i] = time.perf_counter() - inicio
    return tiempos


def medir_memoria_iterar(func, n_iter):
    espacios = np.zeros(n_iter, dtype=int)
    for i in range(n_iter):
        tracemalloc.start()
        func()
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        espacios[i] = peak
    return espacios


def iterar():
    for i in range(10000):
        _ = i

In [ ]:
n_ejecuciones_tiempo = 100
n_ejecuciones_memoria = 50
x_tiempo = np.arange(n_ejecuciones_tiempo)
x_memoria = np.arange(n_ejecuciones_memoria)

In [ ]:
tiempos = medir_tiempo_iterar(iterar, n_ejecuciones_tiempo)
params_tiempo = curve_fit(modelo_constante, x_tiempo, tiempos)[0]
tiempos_ajustados = modelo_constante(x_tiempo, *params_tiempo)

graficar_complejidad(
    x=x_tiempo,
    y_experimental=tiempos,
    y_teorico=tiempos_ajustados,
    nombre_archivo="ciclo_sin_dependencia_tiempo.png",
    ylabel="Tiempo de ejecución [s]",
    funcion="T(n)",
    xlabel="Número de ejecuciones"
)

In [ ]:
recursos = medir_memoria_iterar(iterar, n_ejecuciones_memoria)
params_memoria = curve_fit(modelo_constante, x_memoria, recursos)[0]
recursos_ajustados = modelo_constante(x_memoria, *params_memoria)

graficar_complejidad(
    x=x_memoria,
    y_experimental=recursos,
    y_teorico=recursos_ajustados,
    nombre_archivo="ciclo_sin_dependencia_espacio.png",
    ylabel="Consumo de memoria [bytes]",
    funcion="S(n)",
    xlabel="Número de ejecuciones"
)